In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
import pandas as pd

## Official NREL PVDAQ dataset
url="https://oedi-data-lake.s3.amazonaws.com/pvdaq/csv/systems_20250729.csv"

# Read the CSV into a Pandas DataFrame
df=pd.read_csv(url)

# Display the number of rows and columns
print("Dataset shape:", df.shape)

# Display all column names
print("column names:")
print(df.columns.tolist())

# Display first five rows
display(df.head())

# Check column data types
print("Data types:")
print(df.dtypes)

# Check missing values
print("Missing values:")
print(df.isnull().sum())



In [ ]:
print("Fabric Python is working")

In [ ]:
# Count unique solar systems
print("Total unique systems:", df["system_id"].nunique())

# Check data quality status
print("\nQuality status:")
print(df["qa_status"].value_counts(dropna=False))

# Check missing installed capacity
print("\nMissing capacity:")
print(df["dc_capacity_kW"].isnull().sum())

# Check available sensor channels
print("\nSensor channels:")
print(df["available_sensor_channels"].value_counts().head(10))

# Check dataset sizes
print("\nDataset size statistics (MB):")
print(df["dataset_size_mb"].describe())

# Check whether the five extra columns contain values
print("\nExtra columns:")
print(df.filter(regex="^Unnamed:").notna().sum())

In [ ]:
# Step 1: Create a copy of the original dataset
df_clean=df.copy()

# Step 2: Remove records with missing installed capacity
df_clean=df_clean.dropna(subset=["dc_capacity_kW"])

# Step 3: Filter systems using multiple conditions
df_clean= df_clean[
    (df_clean["qa_status"]=="pass")&
    (df_clean["dc_capacity_kW"]>0)&
    (df_clean["years"]>=3)&
    (df_clean["dataset_size_mb"]>=1)&
    (df_clean["dataset_size_mb"]<=500)&
    (df_clean["available_sensor_channels"]>=3)
]

# Step 4: Check the number of remaining systems
print("Original records:", len(df))
print("Filtered records:"),len(df_clean)

# Step 5: Define the columns we want to inspect
columns_to_show=[
"system_id",
"system_public_name",
"site_location",
"dc_capacity_kW",
"years",
"dataset_size_mb",
"available_sensor_channels",
"qa_status"
]

# Step 6: Display the first 15 systems by dataset size
display(
    df_clean[columns_to_show]
    .sort_values("dataset_size_mb").
    head(15)
)

In [ ]:
# Count total metadata records
print("total records",len(df))

# Count unique system IDs
print("unique system ids",df["system_id"].nunique())

# Count missing system IDs
print("missing system ids",df["system_id"].isna().sum())

# Count duplicated non-null system IDs
duplicate_count=df.loc[
    df["system_id"].notna(),
    "system_id",
].duplicated().sum()

print("duplicate system_ids" , "Duplicate_count")

# Display all records sharing a non-null system ID
duplicate_records=df[
    df["system_id"].notna() &
    df["system_id"].duplicated(keep=False)
]

display(
    duplicate_records[
               ["system_id", "system_public_name", "site_location", "qa_status"]
    ]
)


In [ ]:
# Select the duplicated systems
duplicates = df[df["system_id"].isin([1332, 4901])]

# Check whether all columns match
for system_id, group in duplicates.groupby("system_id"):

    print("System ID:", system_id)

    print("Number of records:", len(group))

    print("Unique complete records:", len(group.drop_duplicates()))

    print("------------------------")

In [ ]:
import requests
from datetime import datetime,timezone
from pathlib import Path

# Original PVDAQ source
source_url = "https://oedi-data-lake.s3.amazonaws.com/pvdaq/csv/systems_20250729.csv"

# Bronze Lakehouse destination
bronze_path = Path("/lakehouse/default/Files/pvdaq/raw")

# Create folders if they do not exist
bronze_path.mkdir(parents=True, exist_ok=True)

# Download the original CSV
response = requests.get(source_url, timeout=120)
response.raise_for_status()

# Save the original file without modifying its contents
file_path = bronze_path / "systems_20250729.csv"

file_path.write_bytes(response.content)

# Record ingestion information
load_time = datetime.now(timezone.utc).isoformat()

print("Bronze file:", file_path)
print("File size (bytes):", file_path.stat().st_size)
print("Load time (UTC):", load_time)

In [ ]:
# Bronze Lakehouse destination
bronze_path=Path("/lakehouse/default/Files/pvdaq/raw")

# Create folders if they do not exist
bronze_path.mkdir(parents=True,exist_ok=True)

# Download the original CSV
response=requests.get(source_url,timeout=120)
response.raise_for_status()

# Save the original file without modifying its contents
file_path = bronze_path / "systems_20250729.csv"
file_path.write_bytes(response.content)

# Record ingestion information
load_time=datetime.now(timezone.utc).isoformat()


print("Bronze file:", file_path)
print("File size (bytes):", file_path.stat().st_size)
print("Load time (UTC):", load_time)

In [ ]:
# Read the original CSV from Bronze Lakehouse
bronze_file="/lakehouse/default/Files/pvdaq/raw/systems_20250729.csv"

df_bronze=pd.read_csv(bronze_file)

# Verify the dataset
print("Rows and columns",df_bronze.shape)
print("unique systems", df_bronze["system_id"].nunique())

# Display first five records
display(df_bronze.head())
